# Neural Networks - Loss Functions

## 1. Why We Need a Loss Function

- How to know a prediction is correct?
- Minimise loss, not get the perfect prediction.
- **Loss landscape** = (weights, loss) -> gradient descent. -> update param
  - weights = where your are
  - loss = altitude
  - gradient = which direction to go downhill
  - optimiser = how to warlk
- Not only tell a prediction is wrong, but also how wrong it is.

```math
\min_{\theta} L(f(x; \theta), y)
```

## 2. Regression Losses

A loss function has 4 properties:
1. error = 0 -> loss = 0
2. error increases -> loss increase
3. loss is never negative
4. loss must be smooth (for gradient descent)

- **MAE**: gradient is 1, -1 -> always sharp like an F1 car with FULL ACCELERATION. While **MSE**'s gradient is $\frac{\partial L}{\partial \^{y}} = 2(\^{y} - y)$, which is gentler (more stable) near the optimum.
- **MSE** penalises error more than **MAE**, but can't be used for very large outliers.
- **MSE** has parabolic shape, **MAE** has sharp corner.
- **Huber Loss** is the best of both, smooth at the centre, then linear edges. For an optimum range $[-\delta, \delta]$, $e = y - \^{y}$

```math
L_{\delta}(e) = \begin{cases}
    \frac{1}{2}e^2, & |e| \le \delta \\
    \delta(|e| - \frac{1}{2}\delta), & |e| > \delta
\end{cases}
```

- **Log-Cosh Loss** also behaves like MSE for small errors, has similar shape to Huber but smoother (I guess?), and MAE for large errors. But unlike Huber, it is smooth everywhere.

```math
L = \log(\cosh(e))
```

| Property                       | MAE               | MSE                 | Huber                       | Log-Cosh                 |
| ------------------------------ | ----------------- | ------------------- | --------------------------- | ------------------------ |
| Smooth                         | ❌                 | ✅                   | Mostly                      | ✅                        |
| Robust to outliers             | ✅                 | ❌                   | ✅                           | ✅                        |
| Large errors punished strongly | ❌                 | ✅                   | Moderate                    | Moderate                 |
| Gradient continuous            | ❌                 | ✅                   | ✅                           | ✅                        |
| Common use                     | Robust regression | Standard regression | Object detection, robust ML | Some deep learning tasks |


In [1]:
import numpy as np

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def huber(y_true, y_pred, delta = 1.0):
    error = y_true - y_pred
    abs_error = np.abs(error)
    quadratic = .5 * error**2
    linear = delta * (abs_error - .5*delta)
    return np.mean(np.where(abs_error <= delta, quadratic, linear))

def log_cosh(y_true, y_pred):
    error = y_true - y_pred
    return np.mean(np.log(np.cosh(error)))

## 3. Probabilistic View of Regression Losses

> Loss functions are not invented. They are discovered from Probability Theory.

- Noise = randomness -> Add random noise into the equation:

```math
y = f(x) + \epsilon
```

- A prediction = probability -> **likelihood**. -> **Maximum Likelihood Estimation (MLE)**.

```math
\theta^* = \arg\max_{\theta} P(\text{data} | \theta)
```

- While Loss requires `min`, likelihood requires `max`.

### Gaussian

> **Assumption**: noise follows Normal Distribution, *large errors are rare, most likely error is zero*.  -> Gaussian probability density

```math
p(y|\^{y}) = \frac{1}{\sqrt{2\pi \sigma^2}}\exp(-\frac{(y - \^{y})^2}{2\sigma^2})
```

- Use $\log(ab) = \log(a) + \log(b)$ to avoid likelihood $P = \prod_i p_i$ to shrink down to zero.
- **Negative Log-Likelihood (NLL)**: optimisers minimise, not maximise -> turn log to negative.
- Substitude Gaussian and remove constants don't depend on the model params -> **Sum of Squared Error (SSE)**.

```math
\log(P) = \sum_i(y_i - \^{y}_i)^2
```

### Laplace

> **Assumption**: noise follows Laplace distribution (sharper peak, heavier tails). *Large errors happen more ofter, most likely error is zero*. -> **Laplace probability density**

```math
p(y|\^{y}) = \frac{1}{2b}\exp(-\frac{|y - \^{y}|}{b})
```

- Take the log again -> **MAE**.

```math
\log(P) = \sum_i |y_i - \^{y}_i|
```

### Bayesian

- Instead of find the best set of params -> estimate **distribution over params**.

> Choose probabilistic model of the world -> Loss follows naturally.

- Papers always start from likelihood -> loss. They ask: "What probabilistic process generated this data?"
  - Gaussian -> MSE
  - Bernoulli -> Binary Cross Entropy
  - Categorical -> Softmax Cross Entropy
  - Poisson -> Poisson loss
  - Negative Binomial -> Count-model losses
- Some hyprid distributions:
  - Mixture distributions (combining multiple probability distributions)
  - Mixture Density Networks (Bishop, 1994)
  - Custom likelihood models
  - Piecewise probability models
  - Robust Bayesian models

## 4. Binary Classification Loss

- Classification -> Probability.
- NNs train through **Gradient**, not Loss.

e.g., Activaiton = Sigmoid. output = .9 -> loss = .1 -> confident. But if output = .1 -> loss = .9 -> *Still confident* but in the opposite class.

Now if we use derivative:

```math
\frac{\partial L}{\partial z} = 2(y - \^{y})\^{y}(1 - \^{y})
```

so if $\^{y}$ reaches 0 or 1 -> derivative reaches 0 (*Gradient saturation*) -> no update.

### Likelihood

- $y = 1 \Rightarrow P(y) = \^{y}$, $y = 0 \Rightarrow P(y) = 1 - \^{y}$. So:

```math
P(y) = \^{y}^y(1 - \^{y})^{1-y}
```

### Binary Cross Entropy

- We want to maximise likelihood = minimise -(likelihood).
- Sum of logs of probability

```math
L = -\log(P) = -[y\log(\^{y}) + (1-y)\log(1 - \^{y})]
```

In [2]:
import numpy as np

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-12 # avoid log(0)
    y_pred = np.clip(y_pred, eps, 1 - eps)

    loss = -(y_true*np.log(y_pred) + (1 - y_true)*np.log(1 - y_pred))
    return np.mean(loss)

In [3]:
y_true = np.array([1, 0, 1, 1])

y_pred = np.array([
    0.9,
    0.2,
    0.8,
    0.4
])

print(binary_cross_entropy(y_true, y_pred))

0.3669845875401002


### What makes BCE trains much faster

e.g., activation = Sigmoid

```math
\frac{\partial L}{\partial z} = \^{y} - y
```

- BCE's derivative contains only error, not sigmoid.

## 5. Multi-class Classification

- For exclusive events: $\sum_i P_i = 1$ -> Can't use Sigmoid.
  - Sigmoid treats each class indepently -> probability sum > 1
- In multi-class, outputs are called **logits**, not probabilities.
- Sum of logits = 1

### Softmax

- Properties:
  - Always positive
  - Smooth
  - Differentiable
  - Large score = large probability
- For logits $z_i$, convert them to $e^{z_i}$, then normalise:

```math
P_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
```

- Exponential = higher score, higher confidence.
- **Probability simplex**: where every probability vector is in.

In [4]:
import numpy as np

def softmax(logits):
    exp = np.exp(logits)
    return exp / np.sum(exp)

### One-hot labels

- **One-hot encoding**: turn classes into an array of 0, 1.

### Categorical Cross Entropy

```math
L = -\sum_i y_i \log p_i
```

where $p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$

e.g., y = [0, 1, 0], prediction: Dog = 0.2, cat = 0.6, horse = 0.2. Cross entropy is: $0*\log(0.2) + 1*\log(0.6) + 0*\log(0.2)$. Everything vanishes except $1*\log(0.6)$.

In [5]:
import numpy as np

def cross_entropy(y_true, y_pred):
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)

    return -np.sum(y_true * np.log(y_pred))

In [6]:
y_true = np.array([0, 1, 0])
y_pred = np.array([0.1, 0.8, 0.1])
loss = cross_entropy(y_true, y_pred)

print(loss)

0.2231435513142097


## 6. Why Cross Entropy Works So Well

- **BCE**'s derivative doesn't have the sigmoid part, while **MSE**'s one saturates to zero.
- Probability -> correctness + confidence.
- **Surprisal**: probability of an impossible event to occur, also $-\log(p)$
- Low probability -> High surprise -> Huge information gain.

> **Entropy**: Average amount of surprise before observing the outcome.

```math
H(P) = -\sum_i P_i \log P_i
```

- Uncertain = High entropy.

> **Cross-entropy** measures how surprised is the model by reality. Training tries to minimise this surprise.

- If a model is *overconfident* -> **calibrate** it.

### KL Divergence

- **Distributional distance** = distance between two Probability distributions. -> **KL Divergence**:

```math
D_{KL}(P||Q) = \sum_i P_i \log \frac{P_i}{Q_i}
```

> **KL Divergence**: How much extra information do I need because I used distribution $Q$ when the true distribution is $P$.

```math
H(P, Q) = H(P) + D_{KL}(P||Q)
```

- $H(P, Q)$: cross-entropy between P and Q.
- $H(P)$: entropy of the true distribution, is constant.
- $D_{KL}(P||Q)$: KL divergence.

> Minimising Cross entropy = Minimising KL Divergence.

## 7. Information Theory Behind Deep Learning

> Training = Learning the probability distribution that generated the data.

> Information depends on **surprise**.

- $P(\text{event}) = 1$ -> the event is perfectly expected -> No information.
- $P(\text{event}) = 0.0000001$ -> NO WAY the event is gonna happen -> Huge information.

### Surprisal

**Surprisal (Self-information)**

```math
I(p)
```

- Properties:
  - Rare event -> More information
  - Indepenent events add information -> $I(x) = - \log P(x)$

e.g., P = 1 -> I = 0 bits, P = 0.5 -> I = 1 bit, P = 0.25 -> I = 2 bits, P = 0.125 -> I = 3 bits.

- $P(A,B) = P(A)P(B)$ but what we want is add (information) $I(A, B) = I(A) + I(B)$. That's why log comes in: $\log(ab) = \log a + \log b$.

e.g., lottery has prob 1 in 10 million -> $-\log_2(10^{-7}) \approx 23.3$ -> Winning tells you 23 bits of information.

### Entropy

> **Assumption**: Don't know the outcome, we have distribution $P(x)$ which tells how much information we expect on average.

```math
H(P) = \sum P(x)(-\log P(x))
```

e.g., coin flip: heads = 99%, tails = 1% -> predictable -> low entropy; heads = tails = 50% -> unpredictable -> high entropy.

### Cross Entropy

> **Cross Entropy**: Total cost to encode reality using my model.

```math
H(P,Q) = -\sum P(x)\log Q(x)
```

- $Q = P$ -> perfect prediction -> cross-entropy = entropy
- Wrong prediction -> cross-entropy increases.

### KL Divergence

```math
D_{KL}(P||Q) = H(P,Q) - H(P) = \sum P(x) \log \frac{P(x)}{Q(x)}
```

- KL divergence is always >= 0. Equality at $P(x) = Q(x)$.
- KL divergence is not distance: $D_{KL}(P||Q) \ne D_{KL}(Q||P)$.
- It measures extra information required because one's beliefs are wrong.

> We don't minimise KL Divergence, we minimise Cross Entropy. Because the $H(P)$ is a fixed constant.

### Coding theory

- Compression machine.
- Common words = short codes. Rare words = long codes.
- Optimal code length: $-\log P(x)$.

### GPT = Compression machine

- Learning probabilities that minimise average code length.
- Every token contributes: $-\log P(token)$.
- Rare words (tokens) = low probability = long code length.

### Why Language Models report Perplexity

> **Perplexity**: Average number of equally likely choices remaining.

```math
2^H
```

- Perplexity = 2 -> roughly 2 possible next tokens
- Perplexity = 100 -> uncertain
- High perplexity -> Worse language model
- Low perplexity -> Better language model


### Summary

> Imagine shortest driving path from London > Edinburgh is 650km
> - **Entropy** = 650km is unavoidable distance, no matter how you travel
> - But you actually drive 730km -> This is **Cross Entropy**.
> - The difference between entropy and cross-entropy is **KL Divergence** = 730 - 650 = 80.

```
Reality
        │
        ▼
Probability Distribution P
        │
        ▼
Entropy
(average uncertainty)
        │
        ▼
Model predicts Q
        │
        ▼
Cross Entropy
        │
        ▼
Cross Entropy
=
Entropy
+
KL Divergence
        │
        ▼
Training
=
Minimise KL
        │
        ▼
Model distribution
≈
Reality
        │
        ▼
Better prediction
        │
        ▼
Better compression
```

## 8. Numerical Stability

## 9. Automatic Differentiation Through Losses

## 10. Loss Landscape

## 11. Modern Loss Functions

## 12. Research Frontier

## 13. Final project